
# Interpretable Machine Learning (IML) Tutorial

This tutorial introduces Interpretable Machine Learning (IML) to help learners understand how and why models make predictions. By the end, participants will appreciate the importance of transparency, trust, and responsible AI, especially in high-stakes domains.


## Overview

In this tutorial we apply several **Interpretable Machine Learning (IML)** techniques to the Wisconsin Breast Cancer dataset. We compare:

| Technique | Scope | Model-agnostic? |
|---|---|---|
| Decision Tree feature importance | Global | No (tree-specific) |
| Random Forest feature importance | Global | No (tree-specific) |
| Permutation Importance | Global | Yes |
| SHAP | Global + Local | Yes (TreeExplainer for trees, KernelExplainer for others) |
| LIME | Local | Yes |

**Why does interpretability matter here?** Cancer diagnosis is high-stakes. A clinician needs to know *why* a model predicts malignant or benign — not just that it does — in order to trust, audit, and safely act on its output.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import shap

# Install lime if not already installed
try:
    from lime.lime_tabular import LimeTabularExplainer
except ModuleNotFoundError:
    !pip install lime
    from lime.lime_tabular import LimeTabularExplainer

plt.rcParams["figure.figsize"] = (8,6)


## Load Dataset

### About the dataset

`sklearn.datasets.load_breast_cancer` loads the **Wisconsin Breast Cancer** dataset (569 samples, 30 numeric features). Each sample is a digitised image of a fine needle aspirate (FNA) of a breast mass. Features describe properties of the cell nuclei such as radius, texture, perimeter, and concavity.

- **Target 0** → malignant  
- **Target 1** → benign

We load it into a Pandas DataFrame so that feature names are preserved throughout the analysis.


In [ ]:

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [ ]:
print(f"Dataset shape: {X.shape}")
print("Target classes: 0 = malignant, 1 = benign\n")

In [ ]:
#list first five
X.head()

In [ ]:
#view columns
X.columns

In [ ]:
#view target variables
y

## Train Test Split

### Why split the data?

We hold out 20 % of samples as a **test set** (`test_size=0.2`) so that all accuracy scores and importance estimates are measured on data the model has *never seen*. Setting `random_state=42` makes the split reproducible. Training on the remaining 80 % gives the models enough examples to learn meaningful patterns from all 30 features.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Building

### Interpretable Model

#### Decision Tree

### Decision Tree, an intrinsically interpretable model

A **Decision Tree** partitions the feature space using a sequence of axis-aligned thresholds (splits). At each internal node the tree asks a yes/no question about one feature (e.g., *"Is worst radius ≤ 16.8?"*). Samples flow left or right until they reach a leaf, which yields the predicted class.

**Why is it interpretable?**  
The full tree can be visualised and each prediction traced along a single root-to-leaf path. No post-hoc explanation technique is needed, the model *is* its own explanation.

**Trade-off:** A single unconstrained tree tends to overfit. We will compare its accuracy and feature rankings with the more powerful (but less transparent) Random Forest.


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

print("Accuracy:", dt.score(X_test, y_test))

#### Feature Importance

#### How Decision Tree feature importance is computed

`dt.feature_importances_` reports **Gini importance** (also called *mean decrease in impurity*). For each feature, sklearn accumulates the weighted reduction in Gini impurity across every node in the tree that splits on that feature. The values are then normalised so they sum to 1.

**Interpretation:** A higher bar means the feature was responsible for larger purity gains across the tree, it was used earlier and/or more frequently to separate the classes.

**Caveat:** Gini importance can overestimate continuous or high-cardinality features because they offer more possible split points. Permutation importance (introduced later) does not share this bias.


In [ ]:
importances = dt.feature_importances_
indices = np.argsort(importances)[-10:]

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.title("Top 10 Features of Decision Tress Classifier")
plt.show()

### Complex Model

####Random Forest

### Random Forest, a powerful but less transparent ensemble

A **Random Forest** trains many decision trees on bootstrap samples of the data, with each tree seeing a random subset of features at every split (*feature bagging*). The final prediction is the majority vote across all trees.

**Why is it harder to interpret?**  
There is no single tree to follow; predictions emerge from hundreds of overlapping trees. This is why we need dedicated explanation tools (permutation importance, SHAP, LIME) to understand what the ensemble has learned.

**Accuracy vs. interpretability:** The Random Forest typically outperforms a single Decision Tree by reducing variance through averaging. The accuracy difference below quantifies this trade-off.


In [ ]:

rf= RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

print("Accuracy:", rf.score(X_test, y_test))


Feature Importance

#### Random Forest feature importance

`rf.feature_importances_` averages the Gini importance of each feature across **all trees** in the forest. Averaging reduces the noise present in any single tree and produces a more stable ranking.

Compare this plot with the Decision Tree importances above. Features that rank highly in *both* are likely to be genuinely important. Features that appear only in one may reflect noise or the idiosyncrasies of a particular tree.


In [ ]:

importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.title("Top 10 Features of raandom forest")
plt.show()



### Exercise 1
- For both the Decision Tree and Random Forest models, which feature appears most important?
- Why do you think these features influence predictions in each model?

### Permutation Importance for Random Forest



### Permutation Importance, a model-agnostic global method

**Core idea:** After training, take the test set and randomly shuffle the values of one feature column. Measure how much the model's accuracy drops. If accuracy falls a lot, that feature was important; if it barely changes, the model did not rely on it.

This is repeated `n_repeats=10` times per feature (with different random shuffles) to get a distribution of importance scores — shown here as a **box plot**. The width of each box reflects how stable the importance estimate is across shuffles.

**Advantages over Gini importance:**
- Works with *any* model (model-agnostic)
- Measured on held-out test data, so it reflects generalisation, not just training fit
- Not biased towards high-cardinality features

**Reading the plot:** Features on the right caused the largest drops in accuracy when shuffled — they are the ones the model relies on most.


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

sorted_idx = perm.importances_mean.argsort()[-10:]

plt.figure(figsize=(10, 6))
plt.boxplot(
    perm.importances[sorted_idx].T,
    vert=False,
    tick_labels=X.columns[sorted_idx]
)
plt.title("Permutation Importance")
plt.xlabel("Decrease in Performance")
plt.tight_layout()
plt.show()

### Exercise 2
- Which features consistently have high permutation importance values across multiple repetitions?
- How does this compare to the feature importance from the Random Forest model?

### SHAP Explanations for Random Forest

### SHAP, SHapley Additive exPlanations

SHAP originates from **cooperative game theory**. Each feature is treated as a "player" and its Shapley value measures its *average marginal contribution* to the prediction across all possible orderings of features.

For **tree-based models**, `shap.TreeExplainer` computes exact Shapley values in polynomial time by exploiting the tree structure — far faster than the model-agnostic KernelExplainer used later for the neural network.

**Summary plot interpretation:**
- Each row is one feature; each dot is one test sample.
- **Position on the x-axis** (SHAP value): positive → pushes the prediction toward *benign*; negative → pushes toward *malignant*.
- **Color**: red = high feature value, blue = low feature value.
- Patterns like "red dots on the right" for a feature mean *high values of that feature increase the benign probability*.

SHAP provides both **global** insight (overall feature ranking by mean |SHAP|) and **local** insight (exact contribution for each individual prediction).


In [ ]:

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Corrected: Select SHAP values for the second class (class index 1) across all samples
shap.summary_plot(shap_values[:, :, 1], X_test)



### Exercise 3
- What do the red and blue colors represent?
- Which features push predictions higher?


### LIME Explanation for Random Forest

### LIME, Local Interpretable Model-agnostic Explanations

Where SHAP explains a model globally and locally using Shapley values, **LIME** focuses purely on *one prediction at a time*.

**How it works:**
1. Take the instance to be explained (`X_test.iloc[0]`).
2. Generate many perturbed versions of that instance by randomly masking or replacing feature values.
3. Ask the black-box model (`rf.predict_proba`) to score each perturbed sample.
4. Fit a simple **linear model** to those scores, weighted so that samples closer to the original instance count more.
5. The linear model's coefficients are the explanation — they show which features pushed the prediction up or down *for this specific instance*.

**Key difference from SHAP:** LIME explanations are *local approximations* and can vary if you rerun them (due to the random perturbation step). SHAP values, by contrast, have a unique mathematical solution. Use LIME when you need a quick, human-readable explanation for a single case; use SHAP when consistency and global coherence matter.


In [ ]:

lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=data.target_names,
    mode='classification'
)

exp = lime_explainer.explain_instance(
    X_test.iloc[0].values,
    rf.predict_proba,
    num_features=10
)

exp.show_in_notebook()



### Exercise 4
- Compare SHAP and LIME outputs.
- Do they highlight the same features?


### Simple Neural Network (MLPClassifier)

---
## Neural Network,  extending IML to deep models

A **Multi-Layer Perceptron (MLPClassifier)** learns non-linear decision boundaries through layers of weighted connections and activation functions. It is a *black-box* model: its internal weights are not directly interpretable.

We now apply the same three post-hoc explanation techniques, permutation importance, SHAP, and LIME, to the neural network and compare the results with those obtained for the Random Forest. This comparison illustrates that IML tools are **model-agnostic**: the same methods work regardless of the underlying model architecture.

`max_iter=1000` is set to ensure the optimiser converges on this dataset.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Ensure X_train, y_train, X_test, y_test are defined for this section
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nn_model = MLPClassifier(random_state=42, max_iter=1000) # Increased max_iter for convergence
nn_model.fit(X_train, y_train)

print("Accuracy:", nn_model.score(X_test, y_test))

#### Permutation Importance for Neural Network

#### Permutation importance for the Neural Network

The procedure is identical to what we did for the Random Forest: shuffle each feature column on the test set and measure the accuracy drop. Because permutation importance only requires calling `model.score()`, it works with *any* sklearn-compatible estimator,  including neural networks.

Comparing this plot with the Random Forest permutation importance reveals whether the two models rely on the same features or have learned different representations of the data.


In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

perm_nn = permutation_importance(
    nn_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

sorted_idx_nn = perm_nn.importances_mean.argsort()[-10:]

plt.figure(figsize=(10, 6))
plt.boxplot(
    perm_nn.importances[sorted_idx_nn].T,
    vert=False,
    tick_labels=X.columns[sorted_idx_nn]
)
plt.title("Permutation Importance for Neural Network")
plt.xlabel("Decrease in Performance")
plt.tight_layout()
plt.show()

#### SHAP Explanations for Neural Network

#### SHAP for the Neural Network, KernelExplainer

`shap.TreeExplainer` only works with tree-based models. For the neural network we use **KernelExplainer**, the model-agnostic variant.

**How KernelExplainer works:**  
It estimates Shapley values by sampling coalitions of features, masking the others with values drawn from a *background dataset* (here `shap.sample(X_train, 100)`), and measuring the change in prediction. It then solves a weighted linear regression to find the Shapley values.

**Trade-off:** KernelExplainer is much slower than TreeExplainer because it cannot exploit model structure. We therefore explain only a 100-sample subset of the test set. In practice, this is a common compromise when applying SHAP to non-tree models.


In [ ]:
import shap
import pandas as pd

# Using a sample of the training data as background for KernelExplainer (can be computationally intensive)
explainer_nn = shap.KernelExplainer(nn_model.predict_proba, shap.sample(X_train, 100))
shap_values_nn = explainer_nn.shap_values(shap.sample(X_test, 100)) # Explaining a sample of X_test

# Corrected: Select SHAP values for the second class (class index 1)
shap.summary_plot(shap_values_nn[:, :, 1], shap.sample(X_test, 100))

#### LIME Explanations for Neural Network

#### LIME for the Neural Network

We reuse the same `LimeTabularExplainer` setup, simply swapping `rf.predict_proba` for `nn_model.predict_proba`. This underlines a key property of LIME: the explainer is **completely decoupled from the model**. Only the prediction function matters.

Check whether the top features identified by LIME for the neural network match those found by SHAP and permutation importance. Systematic disagreement between methods can reveal model instability or features that are correlated (so that swapping one for another barely changes the output).


In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import pandas as pd

lime_explainer_nn = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=data.target_names,
    mode='classification'
)

# Explain a specific instance (e.g., the first instance in X_test)
exp_nn = lime_explainer_nn.explain_instance(
    X_test.iloc[0].values,
    nn_model.predict_proba,
    num_features=10
)

exp_nn.show_in_notebook()

### Exercise 5
- Compare the feature importance, permutation importance, SHAP, and LIME outputs for the Neural Network model.
- Are there any discrepancies between these methods, and if so, what might cause them?


## Quiz Questions

1. What is the difference between global and local interpretability?  
2. Which method (SHAP or LIME) provides global explanations?  
3. Why might two interpretability methods disagree?  
4. What is one limitation of feature importance?  
